# Unit Tests — test_process.py

Tests every function in `src/process.py` using small synthetic data.
Run each cell to verify the data pipeline works correctly.

## Setup

In [18]:
# NOTE: If kagglehub is not installed, run one of these in a Jupyter cell:
#   !pip install kagglehub
#   !pip3 install kagglehub
#   import sys; !{sys.executable} -m pip install kagglehub
import sys
!{sys.executable} -m pip install kagglehub

In [19]:
import numpy as np
import pandas as pd
import sys, os
sys.path.insert(0, '.')
from src.process import (
    clean_data, create_lag_features, create_rolling_features,
    create_interaction_features, get_feature_groups,
    temporal_split, impute_features, prepare_arrays,
)
print("All imports successful")

All imports successful


## Create Synthetic Test Data

In [20]:
# Small fake dataset mimicking real data structure (200 rows, 4 symbols)
np.random.seed(42)
n_rows = 200
n_symbols = 4
rows_per_symbol = n_rows // n_symbols
records = []
for sym in range(n_symbols):
    for i in range(rows_per_symbol):
        row = {
            'date_id': i // 5, 'time_id': i % 5, 'symbol_id': sym,
            'weight': np.random.uniform(0.5, 2.0),
            'responder_6': np.random.normal(0, 1),
            'responder_7': np.random.normal(0, 1),
        }
        for r in range(9):
            if r not in [6, 7]:
                row[f'responder_{r}'] = np.random.normal(0, 1)
        for f in range(10):
            row[f'feature_{f:02d}'] = np.random.normal(0, 1)
        records.append(row)
sample_df = pd.DataFrame(records)

# Also create version with high-null column
sample_df_with_nulls = sample_df.copy()
sample_df_with_nulls['feature_bad'] = np.nan
sample_df_with_nulls.loc[:10, 'feature_bad'] = 1.0

print(f"Synthetic data created: {sample_df.shape}")
print(f"With nulls version: {sample_df_with_nulls.shape}")

Synthetic data created: (200, 23)
With nulls version: (200, 24)


## Test: clean_data() drops high-null columns

In [21]:
cleaned = clean_data(sample_df_with_nulls.copy())
assert 'feature_bad' not in cleaned.columns, "FAIL: feature_bad should be dropped"
print("PASSED — high-null column dropped")


=== Data Cleaning Pipeline ===
BEFORE: 200 rows x 24 columns

Step 1 — Dropped 1 columns (>50% null)
Step 2 — Dropped 7 non-target responders
Step 3 — Sorted by symbol_id, date_id, time_id

All 200 rows preserved. Shape: (200, 16)
PASSED — high-null column dropped


## Test: clean_data() drops non-target responders

In [22]:
cleaned = clean_data(sample_df.copy())
assert 'responder_6' in cleaned.columns, "FAIL: responder_6 should remain"
assert 'responder_7' in cleaned.columns, "FAIL: responder_7 should remain"
assert 'responder_0' not in cleaned.columns, "FAIL: responder_0 should be dropped"
print("PASSED — non-target responders dropped, responder_6 and responder_7 kept")


=== Data Cleaning Pipeline ===
BEFORE: 200 rows x 23 columns

Step 1 — Dropped 0 columns (>50% null)
Step 2 — Dropped 7 non-target responders
Step 3 — Sorted by symbol_id, date_id, time_id

All 200 rows preserved. Shape: (200, 16)
PASSED — non-target responders dropped, responder_6 and responder_7 kept


## Test: clean_data() preserves all rows

In [23]:
cleaned = clean_data(sample_df.copy())
assert len(cleaned) == len(sample_df), f"FAIL: expected {len(sample_df)} rows, got {len(cleaned)}"
print(f"PASSED — all {len(cleaned)} rows preserved")


=== Data Cleaning Pipeline ===
BEFORE: 200 rows x 23 columns

Step 1 — Dropped 0 columns (>50% null)
Step 2 — Dropped 7 non-target responders
Step 3 — Sorted by symbol_id, date_id, time_id

All 200 rows preserved. Shape: (200, 16)
PASSED — all 200 rows preserved


## Test: create_lag_features() creates columns and drops first row per symbol

In [24]:
cleaned = clean_data(sample_df.copy())
n_symbols = cleaned['symbol_id'].nunique()
rows_before = len(cleaned)
result = create_lag_features(cleaned)

assert 'responder_6_lag_1' in result.columns, "FAIL: responder_6_lag_1 missing"
assert 'responder_7_lag_1' in result.columns, "FAIL: responder_7_lag_1 missing"
assert 'responder_7' not in result.columns, "FAIL: responder_7 should be dropped"
assert rows_before - len(result) == n_symbols, f"FAIL: should lose {n_symbols} rows, lost {rows_before - len(result)}"
print(f"PASSED — 2 lag columns created, responder_7 dropped, lost {n_symbols} rows (1 per symbol)")


=== Data Cleaning Pipeline ===
BEFORE: 200 rows x 23 columns

Step 1 — Dropped 0 columns (>50% null)
Step 2 — Dropped 7 non-target responders
Step 3 — Sorted by symbol_id, date_id, time_id

All 200 rows preserved. Shape: (200, 16)

=== Lag Features ===
Created: ['responder_6_lag_1', 'responder_7_lag_1']
Rows: 200 -> 196 (lost 4)
  responder_6_lag_1         correlation: -0.0970
  responder_7_lag_1         correlation: -0.0986
PASSED — 2 lag columns created, responder_7 dropped, lost 4 rows (1 per symbol)


## Test: create_rolling_features() creates columns with roll_ prefix

In [25]:
cleaned = clean_data(sample_df.copy())
result = create_lag_features(cleaned)
rows_before = len(result)
result = create_rolling_features(result, key_features=['feature_00'], windows=[3])

assert 'roll_feature_00_mean_3' in result.columns, "FAIL: roll_feature_00_mean_3 missing"
assert 'roll_feature_00_std_3' in result.columns, "FAIL: roll_feature_00_std_3 missing"
assert len(result) == rows_before, "FAIL: rows should not change"
print(f"PASSED — rolling columns created with roll_ prefix, {len(result)} rows preserved")


=== Data Cleaning Pipeline ===
BEFORE: 200 rows x 23 columns

Step 1 — Dropped 0 columns (>50% null)
Step 2 — Dropped 7 non-target responders
Step 3 — Sorted by symbol_id, date_id, time_id

All 200 rows preserved. Shape: (200, 16)

=== Lag Features ===
Created: ['responder_6_lag_1', 'responder_7_lag_1']
Rows: 200 -> 196 (lost 4)
  responder_6_lag_1         correlation: -0.0970
  responder_7_lag_1         correlation: -0.0986

=== Rolling Features ===
  feature_00 window=3 done
  responder_6_lag_1 window=3 done
PASSED — rolling columns created with roll_ prefix, 196 rows preserved


## Test: create_interaction_features() correct values

In [26]:
cleaned = clean_data(sample_df.copy())
result = create_lag_features(cleaned)
result = create_interaction_features(result)

expected = result['responder_6_lag_1'] * result['responder_7_lag_1']
pd.testing.assert_series_equal(result['resp6_x_resp7'], expected, check_names=False)
print("PASSED — interaction feature = lag_6 * lag_7")


=== Data Cleaning Pipeline ===
BEFORE: 200 rows x 23 columns

Step 1 — Dropped 0 columns (>50% null)
Step 2 — Dropped 7 non-target responders
Step 3 — Sorted by symbol_id, date_id, time_id

All 200 rows preserved. Shape: (200, 16)

=== Lag Features ===
Created: ['responder_6_lag_1', 'responder_7_lag_1']
Rows: 200 -> 196 (lost 4)
  responder_6_lag_1         correlation: -0.0970
  responder_7_lag_1         correlation: -0.0986

=== Interaction Feature ===
  resp6_x_resp7 created
PASSED — interaction feature = lag_6 * lag_7


## Test: get_feature_groups() no overlap

In [27]:
cleaned = clean_data(sample_df.copy())
result = create_lag_features(cleaned)
result = create_rolling_features(result, key_features=['feature_00'], windows=[3])
result = create_interaction_features(result)
groups = get_feature_groups(result)

total = (len(groups['market_cols']) + len(groups['lag_cols']) +
         len(groups['rolling_cols']) + len(groups['interaction_cols']))
assert total == len(groups['all_features']), f"FAIL: sum {total} != total {len(groups['all_features'])}"
print(f"PASSED — no overlap: {total} individual = {len(groups['all_features'])} total")


=== Data Cleaning Pipeline ===
BEFORE: 200 rows x 23 columns

Step 1 — Dropped 0 columns (>50% null)
Step 2 — Dropped 7 non-target responders
Step 3 — Sorted by symbol_id, date_id, time_id

All 200 rows preserved. Shape: (200, 16)

=== Lag Features ===
Created: ['responder_6_lag_1', 'responder_7_lag_1']
Rows: 200 -> 196 (lost 4)
  responder_6_lag_1         correlation: -0.0970
  responder_7_lag_1         correlation: -0.0986

=== Rolling Features ===
  feature_00 window=3 done
  responder_6_lag_1 window=3 done

=== Interaction Feature ===
  resp6_x_resp7 created

=== Feature Summary ===
  Market features:  10
  Lag features:     2
  Rolling features: 4
  Interaction:      1
  ---
  TOTAL:            17
PASSED — no overlap: 17 individual = 17 total


## Test: temporal_split() chronological with no overlap

In [28]:
cleaned = clean_data(sample_df.copy())
result = create_lag_features(cleaned)
train_df, val_df, test_df = temporal_split(result)

train_dates = set(train_df['date_id'].unique())
val_dates = set(val_df['date_id'].unique())
test_dates = set(test_df['date_id'].unique())

assert len(train_dates & val_dates) == 0, "FAIL: train/val dates overlap"
assert len(val_dates & test_dates) == 0, "FAIL: val/test dates overlap"
assert train_df['date_id'].max() < val_df['date_id'].min(), "FAIL: train not before val"
assert val_df['date_id'].max() < test_df['date_id'].min(), "FAIL: val not before test"
print(f"PASSED — chronological order, no date overlap")


=== Data Cleaning Pipeline ===
BEFORE: 200 rows x 23 columns

Step 1 — Dropped 0 columns (>50% null)
Step 2 — Dropped 7 non-target responders
Step 3 — Sorted by symbol_id, date_id, time_id

All 200 rows preserved. Shape: (200, 16)

=== Lag Features ===
Created: ['responder_6_lag_1', 'responder_7_lag_1']
Rows: 200 -> 196 (lost 4)
  responder_6_lag_1         correlation: -0.0970
  responder_7_lag_1         correlation: -0.0986

=== Temporal Split ===
Train: dates 0-7    | 156 rows (80%)
Val:   dates 8-8   | 20 rows (10%)
Test:  dates 9-9  | 20 rows (10%)
PASSED — chronological order, no date overlap


## Test: impute_features() removes all nulls

In [29]:
cleaned = clean_data(sample_df.copy())
result = create_lag_features(cleaned)
result = create_rolling_features(result, key_features=['feature_00'], windows=[3])
result = create_interaction_features(result)
groups = get_feature_groups(result)
train_df, val_df, test_df = temporal_split(result)
train_df, val_df, test_df = impute_features(train_df, val_df, test_df, groups['all_features'])

assert train_df[groups['all_features']].isnull().sum().sum() == 0, "FAIL: train has nulls"
assert val_df[groups['all_features']].isnull().sum().sum() == 0, "FAIL: val has nulls"
assert test_df[groups['all_features']].isnull().sum().sum() == 0, "FAIL: test has nulls"
print("PASSED — zero nulls in train, val, and test")


=== Data Cleaning Pipeline ===
BEFORE: 200 rows x 23 columns

Step 1 — Dropped 0 columns (>50% null)
Step 2 — Dropped 7 non-target responders
Step 3 — Sorted by symbol_id, date_id, time_id

All 200 rows preserved. Shape: (200, 16)

=== Lag Features ===
Created: ['responder_6_lag_1', 'responder_7_lag_1']
Rows: 200 -> 196 (lost 4)
  responder_6_lag_1         correlation: -0.0970
  responder_7_lag_1         correlation: -0.0986

=== Rolling Features ===
  feature_00 window=3 done
  responder_6_lag_1 window=3 done

=== Interaction Feature ===
  resp6_x_resp7 created

=== Feature Summary ===
  Market features:  10
  Lag features:     2
  Rolling features: 4
  Interaction:      1
  ---
  TOTAL:            17

=== Temporal Split ===
Train: dates 0-7    | 156 rows (80%)
Val:   dates 8-8   | 20 rows (10%)
Test:  dates 9-9  | 20 rows (10%)

=== Imputation ===
  Train nulls: 0
  Val nulls:   0
  Test nulls:  0
PASSED — zero nulls in train, val, and test


## Test: prepare_arrays() correct shapes

In [30]:
cleaned = clean_data(sample_df.copy())
result = create_lag_features(cleaned)
result = create_rolling_features(result, key_features=['feature_00'], windows=[3])
result = create_interaction_features(result)
groups = get_feature_groups(result)
train_df, val_df, test_df = temporal_split(result)
train_df, val_df, test_df = impute_features(train_df, val_df, test_df, groups['all_features'])
arrays = prepare_arrays(train_df, val_df, test_df, groups['all_features'])
X_train, y_train, w_train = arrays[0], arrays[1], arrays[2]

assert X_train.shape[1] == len(groups['all_features']), "FAIL: wrong number of features"
assert len(y_train) == X_train.shape[0], "FAIL: y and X row mismatch"
print(f"PASSED — X_train: {X_train.shape}, y_train: {len(y_train)} rows")


=== Data Cleaning Pipeline ===
BEFORE: 200 rows x 23 columns

Step 1 — Dropped 0 columns (>50% null)
Step 2 — Dropped 7 non-target responders
Step 3 — Sorted by symbol_id, date_id, time_id

All 200 rows preserved. Shape: (200, 16)

=== Lag Features ===
Created: ['responder_6_lag_1', 'responder_7_lag_1']
Rows: 200 -> 196 (lost 4)
  responder_6_lag_1         correlation: -0.0970
  responder_7_lag_1         correlation: -0.0986

=== Rolling Features ===
  feature_00 window=3 done
  responder_6_lag_1 window=3 done

=== Interaction Feature ===
  resp6_x_resp7 created

=== Feature Summary ===
  Market features:  10
  Lag features:     2
  Rolling features: 4
  Interaction:      1
  ---
  TOTAL:            17

=== Temporal Split ===
Train: dates 0-7    | 156 rows (80%)
Val:   dates 8-8   | 20 rows (10%)
Test:  dates 9-9  | 20 rows (10%)

=== Imputation ===
  Train nulls: 0
  Val nulls:   0
  Test nulls:  0

=== Final Arrays ===
  X_train: (156, 17)
  X_val:   (20, 17)
  X_test:  (20, 17)
PAS

## Summary

In [31]:
print("=" * 50)
print("ALL test_process.py TESTS PASSED")
print("=" * 50)

ALL test_process.py TESTS PASSED
